# Telecom Customer Churn: Risk Ranking and Retention Economics

This notebook analyzes the `Customer Churn.csv` file in this folder. The supplied data has 3,150 rows and 14 columns; it is **not** the IBM Telco dataset described in the original project outline. In particular, it does not contain contract type, monthly charges, `TotalCharges`, customer identifiers, or add-on services. Numeric codes are kept as codes because no codebook was supplied.

The goal is to create a reproducible baseline, evaluate who a top-risk call list might reach, and make the limits of the available data visible. The Random Forest result is a single stratified holdout result, not a promise of future campaign performance.

## 1. Setup and load

The companion Python module contains reusable loading, feature-engineering, training, and visualization functions. It also writes dashboard-ready figures and model metrics to `output/`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT_DIR = next((folder for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (folder / "Customer Churn.csv").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the project folder containing Customer Churn.csv.")
sys.path.insert(0, str(PROJECT_DIR))
from churn_analysis import load_data, engineer_features, run_analysis

data = load_data(PROJECT_DIR / "Customer Churn.csv")
print(f"Rows: {len(data):,} | Columns: {data.shape[1]} | Churn rate: {data['churn'].mean():.1%}")
display(data.head())

## 2. Data quality and target distribution

Review missingness, duplicate rows, ranges, and the churn base rate. `Subscription Length` is banded as 0–12, 13–24, 25–48, and 49+ **units**; confirm the unit (for example, months) with the data owner before using those labels operationally.

In [ ]:
quality = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing": data.isna().sum(),
    "distinct_values": data.nunique(),
})
display(quality)
print(f"Exact duplicate rows: {data.duplicated().sum():,}")
display(data["churn"].value_counts().rename(index={0: "Not churned", 1: "Churned"}).to_frame("customers"))
display(data.groupby("subscription_length", observed=True)["churn"].agg(["count", "mean"]).head(15))

## 3. Segment checks

Compare rates for available coded categories. These are exploratory associations only: labels and code meanings must be confirmed before translating a code such as `Tariff Plan = 2` into a business recommendation.

In [ ]:
for column in ["complains", "age_group", "tariff_plan", "status", "age"]:
    if column in data:
        print(f"\nChurn by {column} code")
        display(data.groupby(column, dropna=False)["churn"].agg(customers="size", churn_rate="mean").sort_index())

print("\nChurn by subscription-length band")
features = engineer_features(data)
display(pd.DataFrame({"band": features["subscription_length_band"], "churn": data["churn"]}).groupby("band", observed=True)["churn"].agg(customers="size", churn_rate="mean"))

## 4. Train and evaluate

A stratified 80/20 split is fixed at seed 42. The primary models exclude `Status` because its definition and whether it is available before the retention decision are unknown. Logistic Regression and Random Forest use class weighting and preprocessing pipelines. ROC-AUC and average precision are ranking metrics; precision and recall at 0.5 are shown for completeness, while top-30%-capture is the more direct capacity-planning measure.

In [ ]:
data, results = run_analysis()
display(results["metrics"].round(3))
print("Status sensitivity check:", results["status_sensitivity"])
print("\nTop-30% capture means the share of churners in the held-out test set ranked inside the highest-risk 30%.")

### Holdout dashboard and gains chart

In [ ]:
display(Image(filename=str(PROJECT_DIR / "output" / "churn_dashboard.png")))
display(Image(filename=str(PROJECT_DIR / "output" / "cumulative_gains.png")))

## 5. Illustrative campaign economics (not measured revenue)

The CSV has no monthly charge or contract value, so the example below uses explicit assumptions. Replace them with finance-approved values. It assumes holdout capture generalizes to the deployment population and that a successful offer prevents 12 months of revenue loss; neither is established by this observational model. Offer success is causal campaign lift and must be measured with a randomized test/control design.

In [ ]:
monthly_revenue_assumption = 65.00
annual_revenue_months = 12
offer_success_assumption = 0.25
offer_cost_per_contact = 20.00
target_fraction = 0.30

rf_row = results["metrics"].set_index("model").loc["Random Forest"]
estimated_targeted = round(len(data) * target_fraction)
estimated_churners = int(data["churn"].sum())
estimated_captured = estimated_churners * rf_row["top_30_percent_capture"]
estimated_saved = estimated_captured * offer_success_assumption
estimated_retained_revenue = estimated_saved * monthly_revenue_assumption * annual_revenue_months
estimated_campaign_cost = estimated_targeted * offer_cost_per_contact
estimated_net_benefit = estimated_retained_revenue - estimated_campaign_cost
print(f"Estimated customers contacted: {estimated_targeted:,.0f}")
print(f"Estimated churners reached (holdout capture extrapolated): {estimated_captured:,.0f}")
print(f"Estimated saved customers: {estimated_saved:,.0f}")
print(f"Illustrative retained revenue: ${estimated_retained_revenue:,.0f}")
print(f"Illustrative campaign cost: ${estimated_campaign_cost:,.0f}")
print(f"Illustrative net benefit: ${estimated_net_benefit:,.0f}")

## 6. Recommendations and limitations

- Validate the source codebook and the point-in-time meaning of `Status`, `Age`, and `Age Group`; keep only fields available before the call decision.
- Obtain customer IDs and a dated prediction snapshot to create an actionable, auditable call list.
- Validate performance over time and by customer segment; one random holdout does not show drift or generalization to a future campaign.
- Use top-risk ranking as a pilot selection tool, then randomize eligible customers between offer and control groups to estimate incremental retention and ROI.
- This data does not support IBM-specific findings about month-to-month contracts, fiber, electronic checks, autopay, or add-on services. Do not claim those findings from this CSV.